In [ ]:
import os
import sys
from pathlib import Path

library_path = os.path.abspath('../../src')
if library_path not in sys.path:
    sys.path.append(library_path)
library_path = Path(library_path)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

import igraph as ig
import leidenalg
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import kneighbors_graph

DATA_PATH  = library_path.parent / 'data'
PLOTS_PATH = library_path.parent / 'plots'
PLOTS_PATH.mkdir(parents=True, exist_ok=True)

sns.set_theme(style='whitegrid', context='notebook')
pd.set_option('display.float_format', lambda x: f'{x:.3f}')

## Community Detection: Louvain and Leiden Algorithms

### Rationale

K-means and LCA require specifying $K$ in advance and optimise a global objective.
**Graph-based community detection** takes a different approach: construct a similarity
graph over observations and let the algorithm discover the number and shape of communities
without a pre-specified $K$.

The **Louvain** algorithm maximises modularity — the fraction of edges within communities
minus the expected fraction under a null random graph. It is fast and widely used but
has a known resolution limit (merges small communities) and non-deterministic behaviour.

The **Leiden** algorithm is an improved variant that guarantees well-connected communities
and produces more stable, reproducible partitions. It is the preferred method here;
Louvain is run for comparison.

The primary use of community detection in this project is **validation**: do the
communities recovered from the similarity graph align with the classes from LCA
(notebook 02) and the clusters from k-means (notebook 01)? Convergence across methods
strengthens confidence in the subgroup structure.

### Graph construction

A k-nearest-neighbour (kNN) graph is built from the standardised 17-variable feature
matrix. Edge weights are set to the cosine similarity between neighbour pairs.
The choice of $k$ (number of neighbours) controls the resolution of communities;
sensitivity to $k$ is assessed for $k \in \{10, 15, 20, 30\}$.

### Design

- Build kNN similarity graph on the standardised 17-variable feature matrix
- Run Louvain and Leiden; compare number of communities and modularity
- Profile each community: feature means, EQ-5D outcome means, dataset composition
- Cross-tabulate communities vs k-means clusters (notebook 01) and LCA classes
  (notebook 02): compute adjusted Rand index as a convergence metric
- Visualise community structure on the UMAP embedding from notebook 01

In [ ]:
df = pd.read_csv(DATA_PATH / 'wrangled_data.csv', low_memory=False)
print(f'Full dataset: {len(df):,} rows x {df.shape[1]} columns')
print(df['dataset'].value_counts().to_string())

In [ ]:
# 17-variable predictor set (notebook 04)
FEATURES = [
    'Sex', 'age7cat',
    'eth2cat',
    'emp_cat_Employed', 'emp_cat_Other (Sick/Home/etc)', 'emp_cat_Retired',
    'emp_cat_Student', 'emp_cat_Unemployed',
    'edu_cat_2',
    'paVig', 'paMod',
    'smoke_ecig', 'diabetes',
    'meds_num', 'ill_dis',
    'resp', 'skin',
]
FEATURES = [f for f in FEATURES if f in df.columns]
assert len(FEATURES) == 17, f'Expected 17 features, got {len(FEATURES)}'
print(f'Feature set ({len(FEATURES)}): {FEATURES}')